In [20]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores.faiss import FAISS
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain import OpenAI, VectorDBQA
from langchain.chains import RetrievalQAWithSourcesChain
import pandas as pd

In [3]:
# pip install openai
# pip install pypdf
# pip install faiss-cpu


[notice] A new release of pip is available: 23.3.1 -> 23.3.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
loader = DirectoryLoader("sample_documents", loader_cls=PyPDFLoader)
pages = loader.load_and_split()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(pages)

In [43]:
import os
# os.eniron["OPEN_API_KEY"]="" 

embeddings = OpenAIEmbeddings()

db = FAISS.from_documents(docs, embeddings)

In [44]:
def show_vstore(store):
    vector_df = store_to_df(store)
    display(vector_df)

def store_to_df(store):
    v_dict = store.docstore._dict
    data_rows = []
    for k in v_dict.keys():
        doc_name = v_dict[k].metadata['source'].split('/')[-1]
        page_number = v_dict[k].metadata['page']+1
        content = v_dict[k].page_content
        data_rows.append({"chunk_id":k,"document":doc_name,"page":page_number,"content":content})
    vector_df = pd.DataFrame(data_rows)
    return vector_df

save_directory = "sample_vectors"


In [45]:
show_vstore(db)
db.save_local(save_directory, "wales")

,chunk_id,document,page,content
0,b6c7f12d-8e33-4bd1-bd9f-0c9089f24742,sample_documents\wales-in-world.pdf,1,Wales in the World: The Welsh Government’s Int...
1,4945d0f7-8b93-4aee-8963-393a841cd876,sample_documents\wales-in-world.pdf,2,Anglesey Sea Salt Company \nVisiting Wales for...
2,3afcd4a9-7087-48af-b3a0-95c7a337492d,sample_documents\wales-in-world.pdf,3,Contents\nForeword by the First Minister of Wa...
3,f2f1f218-c332-40da-be66-bd4a1db6ca32,sample_documents\wales-in-world.pdf,4,4 Wales in the World: The Welsh Government...
4,1ede02da-6eb0-40da-a0f7-70feb554e4c1,sample_documents\wales-in-world.pdf,4,Active engagement with the European Union (EU)...
5,19208cb7-df1e-4895-9c67-9eb21ad6ab49,sample_documents\wales-in-world.pdf,5,Wales in the World: The Welsh Government’s Int...
6,42f4aa7a-6c95-463d-8b32-ab66620b88ee,sample_documents\wales-in-world.pdf,6,6 Wales in the World: The Welsh Government...
7,aba0a2de-1fc4-42c4-98a4-9dbf7e118dfa,sample_documents\wales-in-world.pdf,7,Wales in the World: The Welsh Government’s Int...
8,adfd101b-ea00-44bd-8afd-dd78e5d88ff3,sample_documents\wales-in-world.pdf,8,8 Wales in the World: The Welsh Government...
9,319608f8-070e-48e0-9248-9f43555d12a4,sample_documents\wales-in-world.pdf,9,Wales in the World: The Welsh Government’s Int...


In [50]:
new_db = FAISS.load_local(save_directory, embeddings)
show_vstore(new_db)

,chunk_id,document,page,content
0,346c719d-4b3a-404a-bce7-4c449f8d1a06,sample_documents\scotland_welcome_to_our_life.pdf,1,Scotland.\nWelcome to our life.\nwww.visitscot...
1,7594fc9e-2ede-4c70-ad58-748f22166249,sample_documents\scotland_welcome_to_our_life.pdf,2,Don’t miss\n1. Explore Scotland’s coastline an...
2,500a0fe4-eebb-4229-a922-1432b23501f2,sample_documents\scotland_welcome_to_our_life.pdf,3,"Celtic Connections, Glasgow Royal Concert Hall..."
3,9089118b-0340-4cd4-ac09-c0ef03c59e58,sample_documents\scotland_welcome_to_our_life.pdf,4,Why not...\n1. Step into the past at the \nhis...
4,a7ae89f9-409d-4224-901b-74d730ae5e19,sample_documents\scotland_welcome_to_our_life.pdf,5,Why not...\n1. Spend a day at Kelvingrove \n...
5,0a5d21bb-a5d0-4c79-b80e-dddb0747326e,sample_documents\scotland_welcome_to_our_life.pdf,6,Hidden gems\n1. Explore the lanes and wynds of...
6,a201825a-703d-409c-a598-47dd1c5e8381,sample_documents\scotland_welcome_to_our_life.pdf,7,P a g e 11 Page 10 MORE ITINERARIES\nWhate...
7,8e5a44fe-7bdc-47e2-b58b-843d768c8944,sample_documents\scotland_welcome_to_our_life.pdf,7,A685A68A68\nA69A96\nA82\nA9\nA84A83A85A828A82\...
8,909f3505-0fdb-43fe-838b-44eab0cbc882,sample_documents\scotland_welcome_to_our_life.pdf,8,"Blairgowrie Farmers’ Market, Perthshire\nChef ..."
9,77e78b81-ccb3-4c12-a0e3-2cada52ac2e3,sample_documents\scotland_welcome_to_our_life.pdf,8,7. Enjoy a delicious and decadent afternoon te...


In [51]:
def delete_document(store, document):
    vector_df = store_to_df(store)
    chunks_list = vector_df.loc[vector_df['document']==document]['chunk_id'].tolist()
    store.delete(chunks_list)


In [63]:
def refresh_model(store):
    retriever = store.as_retriever()
    model = RetrievalQAWithSourcesChain.from_chain_type(llm=OpenAI(), chain_type="stuff", retriever=retriever)
    return model

def questionandanswer(model, question):
    response = model({"question":question}, return_only_outputs=True)
    print("Answer: ", response['answer'])
    print("Sources: ", response['sources'])


question = "Summarize Wales?"

model = refresh_model(new_db)
show_vstore(new_db)
questionandanswer(model, question)

extensions = FAISS.load_local(save_directory, embeddings, "wales")
new_db.merge_from(extensions)
model = refresh_model(new_db)
show_vstore(new_db)
questionandanswer(model, question)

,chunk_id,document,page,content
0,346c719d-4b3a-404a-bce7-4c449f8d1a06,sample_documents\scotland_welcome_to_our_life.pdf,1,Scotland.\nWelcome to our life.\nwww.visitscot...
1,7594fc9e-2ede-4c70-ad58-748f22166249,sample_documents\scotland_welcome_to_our_life.pdf,2,Don’t miss\n1. Explore Scotland’s coastline an...
2,500a0fe4-eebb-4229-a922-1432b23501f2,sample_documents\scotland_welcome_to_our_life.pdf,3,"Celtic Connections, Glasgow Royal Concert Hall..."
3,9089118b-0340-4cd4-ac09-c0ef03c59e58,sample_documents\scotland_welcome_to_our_life.pdf,4,Why not...\n1. Step into the past at the \nhis...
4,a7ae89f9-409d-4224-901b-74d730ae5e19,sample_documents\scotland_welcome_to_our_life.pdf,5,Why not...\n1. Spend a day at Kelvingrove \n...
5,0a5d21bb-a5d0-4c79-b80e-dddb0747326e,sample_documents\scotland_welcome_to_our_life.pdf,6,Hidden gems\n1. Explore the lanes and wynds of...
6,a201825a-703d-409c-a598-47dd1c5e8381,sample_documents\scotland_welcome_to_our_life.pdf,7,P a g e 11 Page 10 MORE ITINERARIES\nWhate...
7,8e5a44fe-7bdc-47e2-b58b-843d768c8944,sample_documents\scotland_welcome_to_our_life.pdf,7,A685A68A68\nA69A96\nA82\nA9\nA84A83A85A828A82\...
8,909f3505-0fdb-43fe-838b-44eab0cbc882,sample_documents\scotland_welcome_to_our_life.pdf,8,"Blairgowrie Farmers’ Market, Perthshire\nChef ..."
9,77e78b81-ccb3-4c12-a0e3-2cada52ac2e3,sample_documents\scotland_welcome_to_our_life.pdf,8,7. Enjoy a delicious and decadent afternoon te...


Answer:   Wales is a country located in the western region of the British Isles. It is bordered by England to the east, the Irish Sea to the north and west, and the Bristol Channel to the south. It has a population of 3.1 million people and a total area of 20,779 square kilometers. Wales is home to a unique culture, language and history, as well as spectacular landscapes and a variety of activities and attractions.

Sources:  N/A


,chunk_id,document,page,content
0,346c719d-4b3a-404a-bce7-4c449f8d1a06,sample_documents\scotland_welcome_to_our_life.pdf,1,Scotland.\nWelcome to our life.\nwww.visitscot...
1,7594fc9e-2ede-4c70-ad58-748f22166249,sample_documents\scotland_welcome_to_our_life.pdf,2,Don’t miss\n1. Explore Scotland’s coastline an...
2,500a0fe4-eebb-4229-a922-1432b23501f2,sample_documents\scotland_welcome_to_our_life.pdf,3,"Celtic Connections, Glasgow Royal Concert Hall..."
3,9089118b-0340-4cd4-ac09-c0ef03c59e58,sample_documents\scotland_welcome_to_our_life.pdf,4,Why not...\n1. Step into the past at the \nhis...
4,a7ae89f9-409d-4224-901b-74d730ae5e19,sample_documents\scotland_welcome_to_our_life.pdf,5,Why not...\n1. Spend a day at Kelvingrove \n...
5,0a5d21bb-a5d0-4c79-b80e-dddb0747326e,sample_documents\scotland_welcome_to_our_life.pdf,6,Hidden gems\n1. Explore the lanes and wynds of...
6,a201825a-703d-409c-a598-47dd1c5e8381,sample_documents\scotland_welcome_to_our_life.pdf,7,P a g e 11 Page 10 MORE ITINERARIES\nWhate...
7,8e5a44fe-7bdc-47e2-b58b-843d768c8944,sample_documents\scotland_welcome_to_our_life.pdf,7,A685A68A68\nA69A96\nA82\nA9\nA84A83A85A828A82\...
8,909f3505-0fdb-43fe-838b-44eab0cbc882,sample_documents\scotland_welcome_to_our_life.pdf,8,"Blairgowrie Farmers’ Market, Perthshire\nChef ..."
9,77e78b81-ccb3-4c12-a0e3-2cada52ac2e3,sample_documents\scotland_welcome_to_our_life.pdf,8,7. Enjoy a delicious and decadent afternoon te...


Answer:   Wales is a small country that has a long history of contact with the wider world. It was an early industrialised nation and has a strong education sector, cultural heritage, and talented people. It is well known for its sports, creative industries, science and innovation, and has established itself as a major centre for the creative industries. Wales is part of the United Kingdom and has to actively engage with the European Union.

Sources:  sample_documents\wales-in-world.pdf


: 

In [5]:
import pickle
vectors = None
# Load the vectors from the pickle file
with open(f"sample_vectors/tour.zip.pkl", "rb") as f:
    vectors = pickle.load(f)
show_vstore(vectors)

        

ModuleNotFoundError: No module named 'openai.api_resources'